# CityFlow Traffic Simulation
### فقط از منوی بالا **Runtime ← Run all** بزنید
---

In [ ]:
# ── مرحله ۱: نصب ──────────────────────────────────────
import os

if not os.path.exists('/content/repo/cityflow.cpython-312-x86_64-linux-gnu.so'):
    print('⏳ Installing CityFlow (3-4 min, only once)...')
    !rm -rf /content/repo
    !git clone https://github.com/persiagfx/cityflow.git /content/repo --quiet
    !bash /content/repo/install.sh
else:
    print('✓ CityFlow already installed — pulling latest updates...')
    !git -C /content/repo pull --quiet
    print('✓ Up to date')

In [ ]:
# ── مرحله ۲: اجرای شبیه‌سازی ──────────────────────────
import sys, os, shutil

sys.path.insert(0, '/content/repo')
os.chdir('/content/repo')

import cityflow

STEPS = 300  # تعداد گام‌های شبیه‌سازی

eng = cityflow.Engine('examples/config.json', thread_num=4)
for i in range(STEPS):
    eng.next_step()

shutil.copy('examples/replay.txt',          'frontend/replay.txt')
shutil.copy('examples/replay_roadnet.json', 'frontend/replay_roadnet.json')

print(f'✓ Simulation done — {STEPS} steps | vehicles: {eng.get_vehicle_count()}')

In [ ]:
# ── مرحله ۳: نمایش ────────────────────────────────────
import subprocess, threading, time, re, os

# HTTP server
def run_server():
    subprocess.run(
        ['python3', '-m', 'http.server', '8765', '--directory', '/content/repo/frontend'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
threading.Thread(target=run_server, daemon=True).start()
time.sleep(1)

# cloudflared tunnel
if not os.path.exists('/content/cloudflared'):
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/content/cloudflared'], check=True)
    subprocess.run(['chmod', '+x', '/content/cloudflared'])

# Write all cloudflared output to a log file — works regardless of stdout/stderr split
logfile = '/tmp/cf.log'
with open(logfile, 'w') as lf:
    proc = subprocess.Popen(
        ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8765'],
        stdout=lf, stderr=lf
    )

print('⏳ Creating link (30 sec)...')
url = None
for _ in range(30):
    time.sleep(1)
    try:
        with open(logfile) as lf:
            content = lf.read()
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', content)
        if match:
            url = match.group(0)
            break
    except Exception:
        pass

print()
print('=' * 55)
print(f'  🌐 {url}')
print('=' * 55)
print()
print('  ① لینک را در Chrome یا Firefox باز کنید')
print('  ② Roadnet File  ←  replay_roadnet.json')
print('  ③ Replay File   ←  replay.txt')
print('  ④ دکمه Start را بزنید')
print()
print('  💡 تا زمانی که این سلول در حال اجراست، لینک فعال است')